# RSNA Knee: сабмит на Kaggle

Делает `submission.csv` из весов, обученных в `02_report_labels_mil.ipynb`. Ничего не учит.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: соревнование `rsna-knee-abnormality-detection` и датасет(ы) с весами `fold*.pth`.
   Можно подключить несколько датасетов (например, ViT-S и ViT-B): ранги всех моделей усредняются.
3. Желательно подключить датасет с декодерами сжатых DICOM (раздел 1), например `goodpjw2008/rsna-knee-2-5d-convnext-reader`.
4. Settings: Accelerator **GPU**, Internet **выключен** (так требует соревнование).
5. Submit. Kaggle перезапустит ноутбук на скрытом test.

Настройки картинок (`IMG`, `CROP_MM`, окна) и модели берутся из файлов весов, поэтому test готовится ровно
так же, как готовился кэш для обучения.

In [ ]:
import os 
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc 
import re
import math 
import time 
import glob 
import json
import sys
import hashlib 
import textwrap 
import warnings
from copy import deepcopy 
from pathlib import Path 
from concurrent.futures import ThreadPoolExecutor 

import cv2 
import numpy as np 
import pandas as pd 
import pydicom 
import matplotlib.pyplot as plt 
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import timm 
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
T0 = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display 
except NameError:
    display = print 

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"устройство: {DEV}, torch {torch.__version__}, timm {timm.__version__}")

## Настройки и веса

In [ ]:
import fnmatch

INPUT = Path("/kaggle/input")
DATA_DIR = INPUT / "competitions" / "rsna-knee-abnormality-detection"
if not DATA_DIR.exists():
    DATA_DIR = INPUT / "rsna-knee-abnormality-detection"
OUT_DIR = Path("/kaggle/working")

SERIES_THREADS = 24        # сколько серий читаем параллельно
EVAL_BATCH = 4
USE_AMP = DEV.type == "cuda"
AMP_DTYPE = torch.float16
TTA_MIRROR = True          # усредняем обычный и зеркальный вид колена
CACHE_MAX_SHARE = 0.55
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# все fold*.pth из подключённых датасетов (в папки соревнования не заходим: там сотни тысяч файлов)
FOLD_FILES = []
for root, dirs, files in os.walk(INPUT):
    dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    FOLD_FILES += [Path(root) / f for f in files if fnmatch.fnmatch(f, "fold*.pth")]
FOLD_FILES = sorted(FOLD_FILES)
assert FOLD_FILES, "Нет весов fold*.pth: подключите датасет с весами в Input"

# настройки подготовки картинок и модели берём из первого файла весов
CFG = torch.load(FOLD_FILES[0], map_location="cpu", weights_only=False)["config"]
IMG, CROP_MM, K_WIN = CFG["IMG"], CFG["CROP_MM"], CFG["K_WIN"]
WIN_BAND = tuple(CFG["WIN_BAND"])
FILL_MISSING_SLOTS = CFG.get("FILL_MISSING_SLOTS", True)
BACKBONE, UNFREEZE_LAST = CFG["BACKBONE"], CFG["UNFREEZE_LAST"]
HIDDEN, TF_DEPTH = CFG["HIDDEN"], CFG["TF_DEPTH"]
WEIGHTS_PATH = False       # предобученные веса не нужны: всё берётся из fold*.pth
for fp in FOLD_FILES:
    log(f"веса: {fp}")
log(f"настройки из весов: IMG={IMG}, CROP_MM={CROP_MM}, K_WIN={K_WIN}, BACKBONE={BACKBONE}")

## 1. Декодеры сжатых DICOM

Часть DICOM-файлов хранится сжатой (JPEG, JPEG-2000, JPEG-LS), и сам `pydicom` их не распакует.
По словам автора ноутбука 0.958, в скрытом test такие файлы есть. Если среди подключённых датасетов есть
`.whl`-файлы декодеров, ячейка ставит их без интернета и проверяет 100 файлов test.

In [ ]:
import importlib, subprocess

wheels = [w for pat in ("/kaggle/input/*/*.whl", "/kaggle/input/*/*/*.whl", "/kaggle/input/*/*/*/*.whl")
          for w in glob.glob(pat) if re.search(r"pylibjpeg|gdcm|jpegls", os.path.basename(w), re.I)]
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
    importlib.invalidate_caches()
log(f"колёс декодеров найдено: {len(wheels)}")

test_series = pd.read_csv(DATA_DIR / "test_series.csv")
n_fail = n_comp = 0
for r in test_series.head(100).itertuples():
    d = DATA_DIR / "test_series" / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
    fs = [f for f in os.listdir(d) if f.endswith(".dcm")] if d.is_dir() else []
    if not fs:
        continue
    try:
        ds = pydicom.dcmread(d / fs[0])
        n_comp += ds.file_meta.TransferSyntaxUID.is_compressed
        ds.pixel_array
    except Exception:
        n_fail += 1
log(f"проверка test: сжатых {n_comp}, не прочитались {n_fail}")

## 2. Слоты, чтение серий и кэш test

Функции те же, что в `03_build_cache_kaggle.ipynb`.

In [ ]:
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]
N_SLOT = len(SLOTS)
SAG_SLOTS = [k for k, (_, plane, _) in enumerate(SLOTS) if plane == "Sagittal"]


def count_files(d):
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0


def build_slot_table(series_df, split):
    # -> таблица «исследование, слот -> какая серия», одна строка на заполненный слот
    df = series_df.copy()
    df["dir"] = [str(DATA_DIR / split / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):
        chosen = {}
        for k, (name, plane, fs) in enumerate(SLOTS):
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                chosen[k] = (cand.sort_values("n_files", ascending=False).iloc[0], False)
        if FILL_MISSING_SLOTS:
            for a, b in ((0, 3), (1, 4), (2, 5)):          # пары «та же плоскость, FS / без FS»
                if a not in chosen and b in chosen:
                    chosen[a] = (chosen[b][0], True)
                elif b not in chosen and a in chosen:
                    chosen[b] = (chosen[a][0], True)
        for k, (r, filled) in chosen.items():
            rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": SLOTS[k][0],
                         "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                         "plane": r["Anatomical_Plane"], "n_files": r["n_files"], "filled": filled})
    return pd.DataFrame(rows)

In [ ]:
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())
        if all(r["ipp"] is not None for r in recs):
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps


def crop_resize(img, ps):
    # квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))
    cw = min(w, int(round(CROP_MM / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def window_centres(n):
    # K_WIN центров окон, равномерно в полосе WIN_BAND, так, чтобы c-1 и c+1 существовали
    a = max(1.0, WIN_BAND[0] * (n - 1))
    b = min(n - 2.0, WIN_BAND[1] * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    return np.clip(np.round(np.linspace(a, b, K_WIN)).astype(int), 0, max(0, n - 1))


def decode(path):
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


DECODE_FAILED = []


def read_series(d, plane_name):
    # -> (uint8 [K_WIN*3, IMG, IMG], позиции окон [K_WIN] от 0 до 1) или None
    recs = read_headers(d)
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)
    n = len(recs)
    centres = window_centres(n)
    idx = np.clip(np.stack([centres - 1, centres, centres + 1], 1).ravel(), 0, n - 1)

    imgs = {}
    for i in sorted(set(idx.tolist())):
        try:
            imgs[i] = decode(os.path.join(d, recs[i]["file"]))
        except Exception:
            imgs[i] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        DECODE_FAILED.append(d)
        return None
    if len(ok) < len(imgs):
        DECODE_FAILED.append(d)
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        a, ps = canon_orient(a, iop, list(ps), ax)
        planes.append((a, ps))

    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [1, 99])
    vol = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    pos = centres / max(n - 1, 1)
    return (vol * 255).round().astype(np.uint8), pos.astype(np.float32)

In [ ]:
def available_gb():
    try:
        with open("/proc/meminfo") as fh:
            info = dict(l.split(":", 1) for l in fh if ":" in l)
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return 16.0


CACHE_KEYS = ("IMG", "CROP_MM", "K_WIN", "WIN_BAND", "FILL_MISSING_SLOTS")


def cache_config():
    # всё, от чего зависят пиксели кэша; должно совпадать при записи и при загрузке
    return {"IMG": IMG, "CROP_MM": CROP_MM, "K_WIN": K_WIN, "WIN_BAND": list(WIN_BAND),
            "FILL_MISSING_SLOTS": FILL_MISSING_SLOTS, "SLOTS": [list(s) for s in SLOTS]}


def build_cache(slot_table, tag, out_dir=None):
    # out_dir=None: кэш в оперативной памяти; out_dir=папка: кэш сразу пишется в файл на диске
    # (так делает 03_build_cache_kaggle.ipynb, чтобы потом сохранить его как датасет)
    global K_WIN
    studies = sorted(slot_table["StudyInstanceUID"].unique())
    need = len(studies) * N_SLOT * K_WIN * 3 * IMG * IMG / 1024 ** 3
    avail = available_gb()
    log(f"{tag}: кэш займёт {need:.1f} ГБ, свободно {avail:.1f} ГБ")
    while out_dir is None and need > CACHE_MAX_SHARE * avail and K_WIN > 2:
        K_WIN -= 1                          # не влезаем - уменьшаем число окон, а не разрешение
        need = len(studies) * N_SLOT * K_WIN * 3 * IMG * IMG / 1024 ** 3
        log(f"   не влезает: K_WIN уменьшен до {K_WIN} ({need:.1f} ГБ)")

    sidx = {s: i for i, s in enumerate(studies)}
    shape = (len(studies), N_SLOT, K_WIN * 3, IMG, IMG)
    if out_dir is None:
        cache = np.zeros(shape, np.uint8)
    else:
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        cache = np.lib.format.open_memmap(out_dir / f"cache_{tag}.npy", mode="w+", dtype=np.uint8, shape=shape)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    pos = np.zeros((len(studies), N_SLOT, K_WIN), np.float32)

    # одна серия может стоять в двух слотах (подстановка) - читаем её один раз
    uniq = slot_table.drop_duplicates("SeriesInstanceUID")[["SeriesInstanceUID", "dir", "plane"]]
    where = slot_table.groupby("SeriesInstanceUID")[["StudyInstanceUID", "slot"]].apply(
        lambda g: list(zip(g["StudyInstanceUID"], g["slot"]))).to_dict()
    jobs = list(uniq.itertuples(index=False))
    log(f"{tag}: читаю {len(jobs)} серий в {SERIES_THREADS} потоков …")
    t0, done = time.time(), 0
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        for c0 in range(0, len(jobs), 256):
            block = jobs[c0:c0 + 256]
            for j, out in zip(block, pool.map(lambda j: read_series(j.dir, j.plane), block)):
                done += 1
                if out is None:
                    continue
                vol, p = out
                for st, k in where[j.SeriesInstanceUID]:
                    cache[sidx[st], k], pos[sidx[st], k], mask[sidx[st], k] = vol, p, 1.0
            if (c0 // 256) % 10 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                log(f"  {tag}: {done}/{len(jobs)} серий, {rate:.1f} серий/с, "
                    f"осталось ≈{(len(jobs) - done) / max(rate, 1e-6) / 60:.0f} мин")
    log(f"{tag}: заполнено слотов {int(mask.sum())}, серий с битыми срезами: {len(DECODE_FAILED)}")
    gc.collect()
    if out_dir is not None:
        cache.flush()
        np.save(out_dir / f"mask_{tag}.npy", mask)
        np.save(out_dir / f"pos_{tag}.npy", pos)
        with open(out_dir / f"studies_{tag}.json", "w") as fh:
            json.dump(studies, fh)
        with open(out_dir / "rsna_cache_meta.json", "w") as fh:
            json.dump(cache_config(), fh, indent=1)
        log(f"{tag}: кэш записан в {out_dir}")
    return studies, cache, mask, pos

## 3. Модель и предсказание

In [ ]:
def load_backbone(name, weights_path, img):
    # weights_path=False -> пустая архитектура (на инференсе веса придут из чекпойнта фолда)
    kw = dict(num_classes=0)                # num_classes=0 -> сеть отдаёт вектор признаков, без головы
    if name.startswith(("vit_", "eva", "beit")):
        kw["img_size"] = img                # у ViT позиционные эмбеддинги зависят от размера входа
    if weights_path is False:
        return timm.create_model(name, pretrained=False, **kw)
    if weights_path and os.path.exists(weights_path):
        m = timm.create_model(name, pretrained=False, **kw)
        sd = torch.load(weights_path, map_location="cpu", weights_only=False)
        for key in ("state_dict", "model"):               # бывают обёртки {"model": {...}}
            if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
                sd = sd[key]
        sd = {k.removeprefix("module.").removeprefix("model."): v for k, v in sd.items()}
        sd = {k: v for k, v in sd.items() if not k.startswith(("head.fc.", "fc.", "classifier."))
              and not (k.startswith("head.") and name.startswith("vit_"))}
        if "pos_embed" in sd and hasattr(m, "pos_embed") and sd["pos_embed"].shape != m.pos_embed.shape:
            # веса DINOv2 сохранены под 518px; под наш IMG интерполируем позиционные эмбеддинги
            from timm.layers import resample_abs_pos_embed
            sd["pos_embed"] = resample_abs_pos_embed(sd["pos_embed"], new_size=m.patch_embed.grid_size,
                                                     num_prefix_tokens=m.num_prefix_tokens)
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not missing and not unexpected, (
            f"веса не подходят к {name}: missing={missing[:5]} unexpected={unexpected[:5]}")
        log(f"энкодер {name}: веса из файла, все ключи совпали")
        return m
    log(f"энкодер {name}: файла весов нет - скачиваю через timm (нужен Internet)")
    return timm.create_model(name, pretrained=True, **kw)


def set_trainable(bb, n_last):
    # всё заморозить, потом открыть «последнюю часть» сети
    for p in bb.parameters():
        p.requires_grad = False
    if hasattr(bb, "blocks"):               # ViT: последние n_last блоков + финальная нормализация
        parts = list(bb.blocks)[-n_last:] + [getattr(bb, "norm", None)]
    elif hasattr(bb, "stages"):             # ConvNeXt и похожие CNN: две последние стадии + голова
        parts = list(bb.stages)[-2:] + [getattr(bb, "head", None), getattr(bb, "norm_pre", None)]
    else:
        parts = [bb]
    for part in parts:
        if part is not None:
            for p in part.parameters():
                p.requires_grad = True


class StudyMIL(nn.Module):
    def __init__(self, backbone=BACKBONE, weights=WEIGHTS_PATH, unfreeze_last=UNFREEZE_LAST,
                 hidden=HIDDEN, tf_depth=TF_DEPTH):
        super().__init__()
        self.backbone = load_backbone(backbone, weights, IMG)
        set_trainable(self.backbone, unfreeze_last)
        d = self.backbone.num_features
        self.proj = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, hidden), nn.LayerNorm(hidden))
        self.slot_emb = nn.Embedding(N_SLOT, hidden)            # «это сагиттальная FS» и т.п.
        self.pos_emb = nn.Sequential(nn.Linear(1, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        if tf_depth:
            layer = nn.TransformerEncoderLayer(hidden, 4, hidden * 2, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.tf = nn.TransformerEncoder(layer, tf_depth, enable_nested_tensor=False)
        else:
            self.tf = None
        # внимание: для каждого окна 12 чисел «насколько это окно важно для патологии c»
        self.att = nn.Sequential(nn.Linear(hidden, 128), nn.Tanh(), nn.Dropout(0.2),
                                 nn.Linear(128, len(TARGETS)))
        self.cls_w = nn.Parameter(torch.randn(len(TARGETS), hidden) * 0.02)   # свой выход у каждой патологии
        self.cls_b = nn.Parameter(torch.zeros(len(TARGETS)))
        # нормализация ImageNet: энкодер учили на входах около N(0, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, mask, pos):
        # x: uint8 [B, S, K, 3, H, W]   mask: [B, S]   pos: [B, S, K]   ->   логиты [B, 12]
        B, S, K = x.shape[:3]
        keep = mask.reshape(-1) > 0.5                          # какие (исследование, слот) есть
        feat = None
        if keep.any():
            xs = x.reshape(B * S, K, *x.shape[3:])[keep].flatten(0, 1)   # только существующие картинки
            xs = (xs.float() / 255.0 - self.mean) / self.std
            f = self.backbone(xs)                              # [n_картинок, d]
            feat = f.new_zeros(B * S, K, f.shape[-1])
            feat[keep] = f.view(-1, K, f.shape[-1])
        else:
            feat = x.new_zeros((B * S, K, self.backbone.num_features), dtype=torch.float32)
        h = self.proj(feat.view(B, S, K, -1))
        h = h + self.slot_emb(torch.arange(S, device=x.device)).view(1, S, 1, -1)
        h = h + self.pos_emb(pos.unsqueeze(-1).to(h.dtype))
        h = h.view(B, S * K, -1)                               # все окна исследования в одну «строку»
        pad = ~(mask > 0.5).unsqueeze(-1).expand(B, S, K).reshape(B, S * K)   # True = окна нет
        pad = pad & ~pad.all(1, keepdim=True)                  # исследование без серий: не маскируем всё
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)   # [B, окна, 12]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)                   # [B, 12, hidden]
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
def augment(x):
    # x: uint8 [..., 3, H, W] на GPU -> та же форма; своё случайное искажение на каждую картинку
    lead = x.shape[:-3]
    y = x.reshape(-1, *x.shape[-3:]).float() / 255.0
    n, dev = y.shape[0], y.device
    rot = (torch.rand(n, device=dev) - 0.5) * 2 * math.radians(AUG_ROT_DEG)
    sc = 1.0 + torch.rand(n, device=dev) * AUG_SCALE            # > 1 -> зум внутрь
    tx = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, y.shape, align_corners=False)
    y = F.grid_sample(y, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    gamma = torch.exp((torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 2 * AUG_GAMMA)   # контраст
    gain = 1.0 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 2 * AUG_GAIN          # яркость
    y = (y.clamp_min(1e-4) ** gamma * gain).clamp(0, 1)
    return (y * 255).round().reshape(*lead, *y.shape[-3:]).to(torch.uint8)


def mirror(x, pos):
    # зеркало колена: x [B, S, K, 3, H, W], pos [B, S, K] -> то же
    x, pos = x.clone(), pos.clone()
    for s in range(x.shape[1]):
        if s in SAG_SLOTS:
            x[:, s] = x[:, s].flip(1).flip(2)        # обратный порядок окон и срезов внутри окна
            pos[:, s] = 1 - pos[:, s].flip(-1)
        else:
            x[:, s] = x[:, s].flip(-1)               # отражение по горизонтали
    return x, pos


def rank_loss(z, y, w):
    parts = []
    for j in range(z.shape[1]):
        zp = z[(y[:, j] > RANK_POS) & (w[:, j] > 0), j]
        zn = z[(y[:, j] < RANK_NEG) & (w[:, j] > 0), j]
        if len(zp) and len(zn):
            parts.append(F.softplus(-(zp[:, None] - zn[None, :])).mean())
    return torch.stack(parts).mean() if parts else z.new_tensor(0.0)


class Ema:
    def __init__(self, model, decay):
        self.decay, self.step = decay, 0
        self.model = deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1.0 + self.step) / (10.0 + self.step))   # разогрев: сначала быстро
        src = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(src[k])


def to_batch(cache, mask, pos, sel, win=None):
    # строки кэша -> тензоры на GPU нужной формы; win - какие окна взять (None = все)
    x = torch.from_numpy(cache[sel]).view(len(sel), N_SLOT, -1, 3, IMG, IMG)
    p = torch.from_numpy(pos[sel])
    if win is not None:
        x, p = x[:, :, win], p[:, :, win]
    return (x.to(DEV, non_blocking=True), torch.from_numpy(mask[sel]).to(DEV), p.to(DEV))


@torch.no_grad()
def predict(model, cache, mask, pos, idx, tta_mirror=TTA_MIRROR):
    # -> вероятности [len(idx), 12]
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = np.sort(idx[b:b + EVAL_BATCH])
        order = np.argsort(np.argsort(idx[b:b + EVAL_BATCH]))   # вернуть исходный порядок
        x, m, p = to_batch(cache, mask, pos, sel)
        with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            prob = torch.sigmoid(model(x, m, p).float())
            if tta_mirror:
                xm, pm = mirror(x, p)
                prob = 0.5 * (prob + torch.sigmoid(model(xm, m, pm).float()))
        out.append(prob.cpu().numpy()[order])
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def macro_auc(y, p):
    # метрика соревнования: среднее AUC по колонкам, где есть оба класса
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j]) if 0 < y[:, j].sum() < len(y)
                             else np.nan for j in range(y.shape[1])]))

## 4. Сабмит

1. Сразу пишем файл со всеми 0.5: если дальше что-то упадёт, на диске всё равно будет валидный сабмит.
2. Строим кэш test в памяти.
3. Каждая модель предсказывает test (обычный и зеркальный вид), предсказания переводятся в ранги и усредняются.
4. Склеиваем с `test.csv` по `StudyInstanceUID`. Исследование без снимков получает 0.5.

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
# сначала пишем «пустой» сабмит со всеми 0.5: если дальше что-то упадёт, файл всё равно будет
_bench = test_df[["StudyInstanceUID"]].copy()
for _t in TARGETS:
    _bench[_t] = 0.5
_bench.to_csv(OUT_DIR / "submission.csv", index=False)

SLOT_TE = build_slot_table(test_series, "test_series")
ST_TE, CTE, MTE, PTE = build_cache(SLOT_TE, "test")

fold_files = FOLD_FILES
ranks = []
for fp in fold_files:
    ck = torch.load(fp, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    assert (cfg["IMG"], cfg["CROP_MM"], cfg["K_WIN"]) == (IMG, CROP_MM, K_WIN), \
        f"{fp.name} обучен с другой подготовкой картинок: {cfg}"
    model = StudyMIL(backbone=cfg["BACKBONE"], weights=False, unfreeze_last=cfg["UNFREEZE_LAST"],
                     hidden=cfg["HIDDEN"], tf_depth=cfg["TF_DEPTH"]).to(DEV)
    model.load_state_dict(ck["model"])
    pr = predict(model, CTE, MTE, PTE, np.arange(len(ST_TE)))
    ranks.append(pd.DataFrame(pr).rank(pct=True).values)
    log(f"{fp.name}: предсказано {len(ST_TE)} исследований")
    del model, ck
    gc.collect()

sub = pd.DataFrame(np.mean(ranks, axis=0), columns=TARGETS)
sub.insert(0, "StudyInstanceUID", ST_TE)
sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
sub[TARGETS] = sub[TARGETS].fillna(0.5)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
log(f"submission.csv: {sub.shape}, пустых значений: {int(sub[TARGETS].isna().sum().sum())}")
display(sub.head())